In [ ]:
# Example 1 — ed25519 sign/verify with CryptoSigner
from securesystemslib.signer import CryptoSigner, Signature, SSlibKey

signer = CryptoSigner.generate_ed25519()
signature = signer.sign(b"hello world")
assert isinstance(signature, Signature)

public_key = signer.public_key
assert isinstance(public_key, SSlibKey)
assert (public_key.keytype, public_key.scheme) == ("ed25519", "ed25519")

# verify_signature returns None on success and raises on mismatch.
public_key.verify_signature(signature, b"hello world")
print("verified with keyid:", public_key.keyid)

In [ ]:
# Example 2 — Key/Signature dict round-trip + tamper detection
from securesystemslib.exceptions import UnverifiedSignatureError
from securesystemslib.signer import CryptoSigner, Signature, SSlibKey

signer = CryptoSigner.generate_ecdsa()
signature = signer.sign(b"payload")

key_dict = signer.public_key.to_dict()
# from_dict consumes (pops fields off) the dict it is handed — pass a copy.
restored = SSlibKey.from_dict(signer.public_key.keyid, dict(key_dict))
assert restored.to_dict() == key_dict

restored_sig = Signature.from_dict(signature.to_dict())
restored.verify_signature(restored_sig, b"payload")

try:
    restored.verify_signature(restored_sig, b"tampered payload")
    raise AssertionError("tampered payload unexpectedly verified")
except UnverifiedSignatureError as e:
    print("OK — tamper rejected:", type(e).__name__)


In [ ]:
# Example 3 — DSSE envelope: PAE, sign, threshold-verify, dict round-trip
from securesystemslib.dsse import Envelope
from securesystemslib.signer import CryptoSigner

signer = CryptoSigner.generate_ed25519()
envelope = Envelope(
    payload=b'{"claim": "artifact is good"}',
    payload_type="application/vnd.in-toto+json",
    signatures={},
)
assert envelope.pae().startswith(b"DSSEv1 ")

envelope.sign(signer)
accepted = envelope.verify(keys=[signer.public_key], threshold=1)
assert signer.public_key.keyid in accepted

restored = Envelope.from_dict(envelope.to_dict())
assert restored.payload == envelope.payload
assert restored.verify(keys=[signer.public_key], threshold=1)
print("envelope verified with", len(accepted), "accepted key(s)")


In [ ]:
# Example 4 — Signer.from_priv_key_uri: the "file2:" private-key-file dispatch path
import pathlib, tempfile
from securesystemslib.signer import CryptoSigner, Signer

signer = CryptoSigner.generate_ed25519()
public_key = signer.public_key

with tempfile.TemporaryDirectory() as tmp:
    key_path = pathlib.Path(tmp) / "ed25519_private.pem"
    key_path.write_bytes(signer.private_bytes)  # unencrypted PEM, per the file2 contract

    loaded = Signer.from_priv_key_uri(f"file2:{key_path}", public_key)
    assert isinstance(loaded, CryptoSigner)
    public_key.verify_signature(loaded.sign(b"round trip"), b"round trip")
print("file2: URI signer round-trip OK")


In [ ]:
# Example 5 — securesystemslib.hash: digest wrappers stay consistent with hashlib
import hashlib, io
import securesystemslib.hash as sslib_hash

data = b"x" * 100_000
h = sslib_hash.digest("sha256")
h.update(data)
assert h.hexdigest() == hashlib.sha256(data).hexdigest()

# File-object digesting reads in chunks — same result as one-shot.
fileobj_digest = sslib_hash.digest_fileobject(io.BytesIO(data), "sha256")
assert fileobj_digest.hexdigest() == h.hexdigest()
print("sha256 =", h.hexdigest()[:32], "... (matches hashlib)")
